# RQ2: preprocess, privately publish, and explore all sentiment datasets

This Colab notebook orchestrates the package preprocessing commands for **AIT V-oc, SST, IMDb, and DynaSent R1/R2**. It then verifies private Hugging Face visibility, reloads the artifacts, and runs descriptive quality checks.

The notebook intentionally keeps preprocessing logic in `sentiment_manifold`; notebook cells only configure, run, and inspect that tested code. SST, IMDb, and DynaSent are correctness-filtered with Pythia-2.8B before equal-length pairing. AIT is not correctness-filtered.

## Before running

1. In Colab, select **Runtime → Change runtime type → GPU**. An L4 or A100 is preferable; a T4 should work with a smaller batch but preprocessing will take longer.
2. When prompted, enter a Hugging Face token with write access. Input is hidden, cached only in notebook memory, and never printed or placed in a command argument.
3. Accept access conditions for any gated tokenizer used below (currently Gemma 2B), or remove that tokenizer from `PAIRING_MODELS`.
4. Download the three official English AIT Valence-oc gold files. The notebook will request them with a Colab upload picker.

**AIT redistribution warning.** The [official Affect in Tweets page](https://www.saifmohammad.com/WebPages/affectintweets.htm) places redistribution restrictions on the data. Private Hub visibility controls access but does not override those terms. AIT upload is therefore disabled unless you explicitly confirm that you have permission; local preprocessing and exploration remain available.

For a reportable run, replace branch names and `None` revisions in the settings cell with immutable commits. The preprocessors record resolved model/tokenizer revisions and source checksums in local metadata.

In [ ]:
# --------------------------- User settings ---------------------------
from pathlib import Path

CONTENT_ROOT = Path("/content")
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = "main"  # Pin a commit SHA for a frozen run.
TIGGES_REFERENCE_URL = "https://github.com/curt-tigges/eliciting-latent-sentiment.git"
TIGGES_REFERENCE_REVISION = "037cbc9c18867e03da14ceace612ab7e0f9449f7"
DYNASENT_URL = "https://github.com/cgpotts/dynasent.git"
DYNASENT_REVISION = "main"  # Pin a commit SHA for a frozen run.

FILTER_MODEL = "pythia-2.8b"
FILTER_REVISION = None  # Optional immutable EleutherAI/pythia-2.8b revision.
FILTER_BATCH_SIZE = 8
PAIRING_MODELS = ["gpt2-small", "qwen-0.6b", "gemma-2b", "pythia-1.4b"]
PAIRING_REVISIONS = {model: None for model in PAIRING_MODELS}
IMDB_REVISION = None  # Optional immutable stanfordnlp/imdb revision.

PUSH_TO_HUB = True
AIT_HUB_UPLOAD_PERMISSION_ACKNOWLEDGED = True # Set to True
REUSE_COMPLETED_LOCAL_OUTPUTS = True
SHOW_TEXT_SAMPLES = False  # Samples are opt-in and use training splits only.

In [ ]:
# Helpers are defined before any setup command so the notebook is restart-safe.
import os
import shlex
import subprocess
import sys

def checked_run(command, *, cwd=None, env=None):
    print("$", shlex.join([str(part) for part in command]))
    subprocess.run(
        [str(part) for part in command], cwd=cwd, env=env, check=True
    )

def command_output(command, *, cwd=None):
    return subprocess.check_output(
        [str(part) for part in command], cwd=cwd, text=True
    ).strip()

def clone_at_revision(url, destination, revision):
    destination = Path(destination)
    if not (destination / ".git").is_dir():
        checked_run(["git", "clone", url, destination])
    if revision:
        checked_run(["git", "checkout", "--detach", revision], cwd=destination)
    resolved = command_output(["git", "rev-parse", "HEAD"], cwd=destination)
    print(f"Resolved {destination.name}: {resolved}")
    return resolved

## 1. Install the project in the Colab runtime

A fresh Colab runtime is expected. Re-running this cell reuses existing clones and checks out the requested revisions without deleting local files.

In [ ]:
PROJECT_ROOT = CONTENT_ROOT / "sentiment-manifold"
PROJECT_COMMIT = clone_at_revision(PROJECT_URL, PROJECT_ROOT, PROJECT_REVISION)
checked_run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_ROOT}[notebooks]"])

In [ ]:
# Authenticate without placing a token literal in the notebook.
import json
from collections import Counter
from functools import lru_cache
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from datasets import get_dataset_config_names, load_dataset, load_from_disk
from huggingface_hub import HfApi
from IPython.display import display
from google.colab import files
from getpass import getpass

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): ").strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def delete_runtime_secret(name):
    value = _RUNTIME_SECRETS.pop(name, None)
    if value is not None:
        del value

_token = get_runtime_secret("HF_TOKEN")
HF_ACCOUNT = HfApi(token=_token).whoami()["name"]
del _token
print("Hugging Face account:", HF_ACCOUNT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: Pythia-2.8B filtering will be very slow without a GPU.")

In [ ]:
# Private dataset destinations are derived from the authenticated account.
OUTPUTS = {
    "ait": PROJECT_ROOT / "data/processed/ait-valence-binary",
    "sst": PROJECT_ROOT / "data/processed/sst-pythia-2.8b",
    "imdb": PROJECT_ROOT / "data/processed/imdb-pythia-2.8b",
    "dynasent": PROJECT_ROOT / "data/processed/dynasent-r1-r2-pythia-2.8b",
}
REPOS = {
    "ait": f"{HF_ACCOUNT}/sentiment-manifold-ait-valence-binary",
    "sst": f"{HF_ACCOUNT}/sentiment-manifold-sst-pythia-2.8b",
    "imdb": f"{HF_ACCOUNT}/sentiment-manifold-imdb-pythia-2.8b",
    "dynasent": f"{HF_ACCOUNT}/sentiment-manifold-dynasent-r1-r2-pythia-2.8b",
}
SHOULD_PUSH = {
    key: PUSH_TO_HUB and (key != "ait" or AIT_HUB_UPLOAD_PERMISSION_ACKNOWLEDGED)
    for key in OUTPUTS
}
display(pd.DataFrame({"local_output": OUTPUTS, "hub_repo": REPOS, "will_push": SHOULD_PUSH}))

## 2. Acquire and validate source data

SST is read from the commit-pinned Tigges et al. reference repository. DynaSent is extracted from the official repository archive. IMDb is downloaded by the package from Hugging Face. AIT must be obtained under its source terms and uploaded into the runtime by the researcher.

In [ ]:
REFERENCE_ROOT = CONTENT_ROOT / "eliciting-latent-sentiment"
DYNASENT_REPOSITORY = CONTENT_ROOT / "dynasent-source"
TIGGES_REFERENCE_COMMIT = clone_at_revision(
    TIGGES_REFERENCE_URL, REFERENCE_ROOT, TIGGES_REFERENCE_REVISION
)
DYNASENT_COMMIT = clone_at_revision(
    DYNASENT_URL, DYNASENT_REPOSITORY, DYNASENT_REVISION
)

SST_ROOT = REFERENCE_ROOT / "stanfordSentimentTreebank"
DYNASENT_ROOT = CONTENT_ROOT / "data/raw/dynasent"
DYNASENT_ROOT.mkdir(parents=True, exist_ok=True)
dynasent_archives = list(DYNASENT_REPOSITORY.rglob("dynasent-v1.1.zip"))
if len(dynasent_archives) != 1:
    raise FileNotFoundError("Expected exactly one dynasent-v1.1.zip in the official repository")
with zipfile.ZipFile(dynasent_archives[0]) as archive:
    archive.extractall(DYNASENT_ROOT)
print("SST source:", SST_ROOT)
print("DynaSent source:", DYNASENT_ROOT)

In [ ]:
# Upload the official English AIT Valence-oc train, dev, and test-gold files.
# You may instead set AIT_ROOT to a mounted Drive directory containing those files.
AIT_ROOT = CONTENT_ROOT / "data/raw/ait"
AIT_ROOT.mkdir(parents=True, exist_ok=True)

from sentiment_manifold.data.preprocessing.ait import discover_ait_files

try:
    AIT_FILES = discover_ait_files(AIT_ROOT)
except FileNotFoundError:
    print("Select the three official AIT Valence-oc English files (or an archive containing them).")
    uploaded = files.upload()
    for name, payload in uploaded.items():
        destination = AIT_ROOT / Path(name).name
        destination.write_bytes(payload)
        if zipfile.is_zipfile(destination):
            with zipfile.ZipFile(destination) as archive:
                archive.extractall(AIT_ROOT)
    AIT_FILES = discover_ait_files(AIT_ROOT)

print("Discovered AIT splits:")
for split, path in AIT_FILES.items():
    print(f"  {split}: {path.name}")

In [ ]:
# Fail early if source discovery is incomplete. No preprocessing has run yet.
from sentiment_manifold.data.preprocessing.dynasent import discover_dynasent_files

if not SST_ROOT.is_dir():
    raise FileNotFoundError(SST_ROOT)
DYNASENT_FILES = discover_dynasent_files(DYNASENT_ROOT, rounds=(1, 2))
assert set(AIT_FILES) == {"train", "dev", "test"}
assert len(DYNASENT_FILES) == 6
print("Source validation passed: AIT=3 splits, DynaSent=6 round/split files, SST present.")

## 3. Preprocess and publish

Each command writes one local directory containing named Hugging Face `DatasetDict` configurations, `metadata.json`, and a dataset card. If publishing is enabled for that dataset, every configuration is pushed to the specified **private** repository.

The Pythia-2.8B process exits between SST, IMDb, and DynaSent, releasing GPU memory. Model downloads remain cached by Colab.

In [ ]:
def pairing_arguments():
    arguments = []
    for model in PAIRING_MODELS:
        arguments.extend(["--pairing-model", model])
        revision = PAIRING_REVISIONS.get(model)
        if revision:
            arguments.extend(["--pairing-revision", f"{model}={revision}"])
    return arguments

def filter_arguments():
    arguments = [
        "--filter-model", FILTER_MODEL,
        "--device", "auto",
        "--dtype", "auto",
        "--batch-size", str(FILTER_BATCH_SIZE),
    ]
    if FILTER_REVISION:
        arguments.extend(["--filter-revision", FILTER_REVISION])
    return arguments

def publish_arguments(dataset_key):
    if not SHOULD_PUSH[dataset_key]:
        return []
    return ["--push-to-hub", "--private", "--hub-repo-id", REPOS[dataset_key]]

def run_preprocessor(dataset_key, subcommand, arguments):
    metadata_path = OUTPUTS[dataset_key] / "metadata.json"
    if REUSE_COMPLETED_LOCAL_OUTPUTS and metadata_path.is_file():
        print(f"Reusing completed local artifact: {OUTPUTS[dataset_key]}")
        print("Set REUSE_COMPLETED_LOCAL_OUTPUTS=False and remove that directory to regenerate.")
        return
    command = [
        sys.executable, "-m", "sentiment_manifold.cli", subcommand,
        "--output-dir", OUTPUTS[dataset_key],
        *arguments,
        *pairing_arguments(),
        *publish_arguments(dataset_key),
    ]
    child_env = os.environ.copy()
    child_env["HF_TOKEN"] = get_runtime_secret("HF_TOKEN")
    child_env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    try:
        checked_run(command, cwd=PROJECT_ROOT, env=child_env)
    finally:
        child_env.pop("HF_TOKEN", None)
        del child_env

In [ ]:
# AIT: preserve ordinal valence, exclude zero, map negative/positive classes, and pair.
# Deliberately no correctness-filter arguments are used for AIT.
if PUSH_TO_HUB and not AIT_HUB_UPLOAD_PERMISSION_ACKNOWLEDGED:
    print("AIT will remain local because upload permission was not acknowledged.")
run_preprocessor("ait", "preprocess-ait", ["--ait-root", AIT_ROOT])

In [ ]:
# SST: build both label-policy variants; only test is Pythia-filtered and paired.
run_preprocessor(
    "sst",
    "preprocess-sst",
    ["--sst-root", SST_ROOT, "--binarization", "both", *filter_arguments()],
)

In [ ]:
# IMDb: retain source labels/text; only test is Pythia-filtered and paired by default.
imdb_arguments = ["--dataset-name", "stanfordnlp/imdb", *filter_arguments()]
if IMDB_REVISION:
    imdb_arguments.extend(["--dataset-revision", IMDB_REVISION])
run_preprocessor("imdb", "preprocess-imdb", imdb_arguments)

In [ ]:
# DynaSent: keep rounds separate; only gold positive/negative test records are filtered/paired.
run_preprocessor(
    "dynasent",
    "preprocess-dynasent",
    ["--dynasent-root", DYNASENT_ROOT, *filter_arguments()],
)

In [ ]:
# Summarize local provenance before examining any examples.
LOCAL_METADATA = {}
provenance_rows = []
for key, output_dir in OUTPUTS.items():
    metadata_path = output_dir / "metadata.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing completed metadata for {key}: {metadata_path}")
    metadata = json.loads(metadata_path.read_text())
    LOCAL_METADATA[key] = metadata
    correctness = metadata.get("correctness_filter", {})
    provenance_rows.append({
        "dataset": key,
        "filter": correctness.get("filter_model_alias", "none"),
        "filter_revision": correctness.get("resolved_filter_model_revision"),
        "pairing_models": ", ".join(metadata.get("pairing_models", [])),
        "configs": len(metadata.get("dataset_configs", [])),
    })
PROVENANCE_TABLE = pd.DataFrame(provenance_rows)
display(PROVENANCE_TABLE)

In [ ]:
# Verify that every requested Hub destination exists and is private.
privacy_rows = []
for key, should_push in SHOULD_PUSH.items():
    if not should_push:
        privacy_rows.append({"dataset": key, "repo": REPOS[key], "status": "local only"})
        continue
    info = HfApi(token=get_runtime_secret("HF_TOKEN")).repo_info(
        REPOS[key], repo_type="dataset"
    )
    if info.private is not True:
        raise RuntimeError(f"Expected a private dataset repository: {REPOS[key]}")
    privacy_rows.append({"dataset": key, "repo": REPOS[key], "status": "private"})
PRIVACY_TABLE = pd.DataFrame(privacy_rows)
display(PRIVACY_TABLE)

## 4. Reload and explore the processed artifacts

The following cells read from the private Hub repositories when published and otherwise fall back to the local saved artifact (normally AIT). These are **descriptive preprocessing checks**, not evidence of a causal mechanism. Do not use the locked test results below to choose layers, fitters, intervention strengths, or other hyperparameters.

In [ ]:
def available_configs(dataset_key):
    if SHOULD_PUSH[dataset_key]:
        return get_dataset_config_names(
            REPOS[dataset_key], token=get_runtime_secret("HF_TOKEN")
        )
    return sorted(path.name for path in OUTPUTS[dataset_key].iterdir() if path.is_dir())

@lru_cache(maxsize=None)
def load_processed(dataset_key, config_name):
    if SHOULD_PUSH[dataset_key]:
        return load_dataset(
            REPOS[dataset_key], config_name, token=get_runtime_secret("HF_TOKEN")
        )
    return load_from_disk(OUTPUTS[dataset_key] / config_name)

CONFIGS = {key: available_configs(key) for key in OUTPUTS}
config_rows = [
    {"dataset": key, "configuration": config}
    for key, configs in CONFIGS.items()
    for config in configs
]
CONFIG_TABLE = pd.DataFrame(config_rows)
display(CONFIG_TABLE.groupby("dataset").agg(
    configuration_count=("configuration", "count"),
    configurations=("configuration", lambda values: ", ".join(values)),
))

### Label balance

This checks the ground-truth binary labels in each retained split. It does not treat the labels as model predictions. Large imbalances should be recorded because they affect fitters and uncertainty estimates.

In [ ]:
BINARY_CONFIGS = {
    "ait": ["binary"],
    "sst": ["tigges_binarized", "neutral_removed_binarized"],
    "imdb": ["binary"],
    "dynasent": ["r1_binary", "r2_binary"],
}
label_rows = []
for dataset_key, config_names in BINARY_CONFIGS.items():
    for config_name in config_names:
        artifact = load_processed(dataset_key, config_name)
        for split, records in artifact.items():
            counts = Counter(int(label) for label in records["label"])
            for label, label_name in ((0, "negative"), (1, "positive")):
                label_rows.append({
                    "dataset": dataset_key, "configuration": config_name,
                    "split": split, "label": label_name, "rows": counts[label],
                })
LABEL_TABLE = pd.DataFrame(label_rows)
display(LABEL_TABLE.pivot_table(
    index=["dataset", "configuration", "split"],
    columns="label", values="rows", fill_value=0,
))

### Pythia-2.8B correctness-gate retention

For SST, IMDb, and DynaSent, labels stay equal to dataset ground truth. The binary labels only determine whether Pythia's larger `Positive`/`Negative` logit agrees with the example; they never relabel it. The retention rate below is therefore a preprocessing audit, not a directional-patching metric.

In [ ]:
GATE_CONFIGS = [
    ("sst", "tigges_pythia_scored", "tigges_pythia_correct", "test"),
    ("sst", "neutral_removed_pythia_scored", "neutral_removed_pythia_correct", "test"),
    ("imdb", "pythia_scored", "pythia_correct", "test"),
    ("dynasent", "r1_pythia_scored", "r1_pythia_correct", "test"),
    ("dynasent", "r2_pythia_scored", "r2_pythia_correct", "test"),
]
gate_rows = []
for dataset_key, scored_config, correct_config, split in GATE_CONFIGS:
    scored = len(load_processed(dataset_key, scored_config)[split])
    correct = len(load_processed(dataset_key, correct_config)[split])
    gate_rows.append({
        "dataset": dataset_key, "configuration": scored_config.removesuffix("_pythia_scored"),
        "scored": scored, "retained_correct": correct,
        "retention_percent": 100.0 * correct / scored if scored else float("nan"),
    })
GATE_TABLE = pd.DataFrame(gate_rows)
display(GATE_TABLE.round(2))
sns.barplot(data=GATE_TABLE, x="retention_percent", y="configuration", hue="dataset")
plt.xlim(0, 100)
plt.xlabel("Pythia-correct records retained (%)")
plt.ylabel("")
plt.title("Correctness-filter retention (preprocessing QC only)")
plt.show()

### Equal-length pair coverage for GPT-2, Qwen, and the common intersection

A matched pair contains one positive and one negative prompt with equal **full-prompt** length for the named tokenizer. `common` requires equality for all selected tokenizers. Its count should therefore be no larger than any individual-model count.

In [ ]:
PAIR_DATASETS = [
    ("ait", "AIT", ""),
    ("sst", "SST (Tigges labels)", "tigges_"),
    ("imdb", "IMDb", ""),
    ("dynasent", "DynaSent R1", "r1_"),
    ("dynasent", "DynaSent R2", "r2_"),
]
PAIRING_PREFIXES = {"GPT-2 Small": "gpt2_small", "Qwen3 0.6B": "qwen_0_6b", "Common": "common"}
pair_rows = []
for dataset_key, display_name, dataset_prefix in PAIR_DATASETS:
    for model_name, model_prefix in PAIRING_PREFIXES.items():
        config_name = f"{dataset_prefix}{model_prefix}_matched_pairs"
        artifact = load_processed(dataset_key, config_name)
        pair_rows.append({
            "dataset": display_name, "pairing": model_name,
            "matched_pairs": sum(len(records) for records in artifact.values()),
        })
PAIR_TABLE = pd.DataFrame(pair_rows)
display(PAIR_TABLE.pivot(index="dataset", columns="pairing", values="matched_pairs"))
sns.barplot(data=PAIR_TABLE, x="matched_pairs", y="dataset", hue="pairing")
plt.xlabel("Matched positive/negative pairs")
plt.ylabel("")
plt.title("Equal-full-prompt-length pair coverage")
plt.show()

### Directed-pair invariant checks

Every matched pair must expand to exactly two causal cases: negative receiver with positive donor, and positive receiver with negative donor. Labels define those roles; the eventual logit-flip and normalized logit-difference metrics must be computed from model outputs after intervention, not from the labels themselves.

In [ ]:
invariant_rows = []
for dataset_key, display_name, dataset_prefix in PAIR_DATASETS:
    for model_name, model_prefix in PAIRING_PREFIXES.items():
        config_name = f"{dataset_prefix}{model_prefix}_directed_pairs"
        artifact = load_processed(dataset_key, config_name)
        case_count = 0
        pair_counts = Counter()
        signatures = {}
        for records in artifact.values():
            length_columns = [name for name in records.column_names if name.endswith("_prompt_num_tokens")]
            if not length_columns:
                raise AssertionError(f"No prompt-length field in {config_name}")
            for row in records:
                if int(row["source_label"]) == int(row["target_label"]):
                    raise AssertionError(f"Same-label causal pair in {config_name}")
                pair_id = row["pair_id"]
                signature = tuple(int(row[name]) for name in length_columns)
                if pair_id in signatures and signatures[pair_id] != signature:
                    raise AssertionError(f"Length signature changed within {pair_id}")
                signatures[pair_id] = signature
                pair_counts[pair_id] += 1
                case_count += 1
        if any(count != 2 for count in pair_counts.values()):
            raise AssertionError(f"A directed pair did not have two cases in {config_name}")
        invariant_rows.append({
            "dataset": display_name, "pairing": model_name,
            "matched_pairs": len(pair_counts), "directed_cases": case_count, "status": "passed",
        })
INVARIANT_TABLE = pd.DataFrame(invariant_rows)
display(INVARIANT_TABLE)

### AIT ordinal valence distribution (development data only)

AIT's `original_valence_class` is an ordered category from −3 to +3, not an interval-scale continuous target. Zero is excluded by the binary policy. Only train and validation are shown so the final test split remains locked for confirmation.

In [ ]:
ait_binary = load_processed("ait", "binary")
ait_valence_rows = []
for split in ("train", "validation"):
    counts = Counter(int(value) for value in ait_binary[split]["original_valence_class"])
    for ordinal_class, rows in sorted(counts.items()):
        ait_valence_rows.append({"split": split, "ordinal_class": ordinal_class, "rows": rows})
AIT_VALENCE_TABLE = pd.DataFrame(ait_valence_rows)
display(AIT_VALENCE_TABLE)
sns.barplot(data=AIT_VALENCE_TABLE, x="ordinal_class", y="rows", hue="split")
plt.xlabel("Original ordinal valence class")
plt.ylabel("Records")
plt.title("AIT V-oc development distribution (zero excluded)")
plt.show()

In [ ]:
# Optional schema/text inspection. It is off by default and samples only training data.
# This is useful for formatting checks, not for choosing models or interventions.
if SHOW_TEXT_SAMPLES:
    sample_specs = [
        ("ait", "binary"), ("sst", "tigges_binarized"),
        ("imdb", "binary"), ("dynasent", "r1_binary"),
    ]
    for dataset_key, config_name in sample_specs:
        records = load_processed(dataset_key, config_name)["train"]
        row = records.shuffle(seed=42).select([0])[0]
        print(f"\n{dataset_key} / {config_name}")
        print("columns:", records.column_names)
        print("label:", row["label"], "text preview:", row["text"][:240])
else:
    print("Text sampling is disabled. Set SHOW_TEXT_SAMPLES=True to inspect fixed-seed training examples.")

In [ ]:
# Save small, shareable summaries. These CSVs contain counts/provenance only, not source text.
SUMMARY_DIR = CONTENT_ROOT / "rq2-data-exploration"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary_tables = {
    "provenance.csv": PROVENANCE_TABLE,
    "privacy.csv": PRIVACY_TABLE,
    "configurations.csv": CONFIG_TABLE,
    "label_counts.csv": LABEL_TABLE,
    "correctness_gate.csv": GATE_TABLE,
    "pair_coverage.csv": PAIR_TABLE,
    "pair_invariants.csv": INVARIANT_TABLE,
    "ait_development_valence_counts.csv": AIT_VALENCE_TABLE,
}
for filename, table in summary_tables.items():
    table.to_csv(SUMMARY_DIR / filename, index=False)
print("Saved exploration summaries to", SUMMARY_DIR)

## Remove the Hugging Face token from memory

Run the next cell when publishing and private-data exploration are complete—or at any time you want to forget the cached credential. A later Hub operation will prompt for the token again.

In [ ]:
delete_runtime_secret("HF_TOKEN")
os.environ.pop("HF_TOKEN", None)
print("Removed HF_TOKEN from the notebook secret cache and process environment.")

## Interpretation boundary

Passing these checks establishes dataset provenance, label counts, correctness-filter retention, and equal-length causal-pair structure. It does **not** establish that a learned direction is causal. Direction fitting must use training activations, layer/hyperparameter selection must use validation data, and final test/OOD evaluation must stay locked until confirmation. Causal conclusions additionally require directional patching and controls.